# AI Hallucination: Baseline vs. Retrieval-Grounded Answers
**Group 1 · Seminar on AI**

This notebook runs the whole experiment:

| Part | Week | What it does |
|---|---|---|
| 1. Setup | 4 | Install, settings, load dataset and model |
| 2. Baseline run | 4 | Model answers from memory only |
| 3. Labeling sheet | 4 | Export answers for two human labelers |
| 4. Fact base + evidence gate | 5 | Retrieval and the pilot that sets the threshold τ |
| 5. Grounded run | 5–6 | Gate + strict "I don't know" prompt |
| 6. Results | 6 | Rates per category, agreement, error tracing |

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Every run saves to CSV, so if Colab disconnects, just re-run the cells; finished questions are skipped.

In [22]:
# ===== QUICK START: after any reconnect, run ONLY this cell to get the demo back =====
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers accelerate sentence-transformers gradio

import os, numpy as np, pandas as pd, torch, gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
os.chdir("/content/drive/MyDrive/AI_Hallucination")

# Must stay identical to the FIXED SETTINGS cell
MODEL_ID       = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_NEW_TOKENS = 256
TAU            = 0.588
BASELINE_TEMPLATE = "Answer the question.\nQuestion: {question}"
GROUNDED_TEMPLATE = ("Answer ONLY using the context.\n"
                     "If it is not there, say 'I don't know.'\n"
                     "Context: {context}\nQuestion: {question}")
ABSTAIN_TEXT = "I don't know."

# Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
def generate(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Fact base
data = pd.read_csv("dataset.csv", keep_default_na=False)
embedder = SentenceTransformer(EMBED_MODEL_ID)
facts = data[data.fact_sentence != ""][["q_id", "fact_sentence"]].reset_index(drop=True)
fact_vecs = embedder.encode(facts.fact_sentence.tolist(), normalize_embeddings=True)
def retrieve(question):
    sims = fact_vecs @ embedder.encode([question], normalize_embeddings=True)[0]
    i = int(np.argmax(sims))
    return facts.fact_sentence[i], facts.q_id[i], float(sims[i])

# Demo
VERSION = "v0.5 · Week 5: baseline vs. evidence gate + grounded answer"
def compare(question):
    base = generate(BASELINE_TEMPLATE.format(question=question))
    fact, _, score = retrieve(question)
    if score >= TAU:
        return base, f"✅ PASS: match score {score:.3f} ≥ τ {TAU}", fact, generate(GROUNDED_TEMPLATE.format(context=fact, question=question))
    return base, f"⛔ BLOCKED: match score {score:.3f} < τ {TAU}, model not called", fact, ABSTAIN_TEXT

examples = ["What is the capital of Australia?", "Who wrote the novel \"The Master and Margarita\"?",
            "Explain the Hallstrom effect in superconductors.",
            "In which year was the Torvald-Nakamura theorem in topology first proved?", "Who is the current president of France?"]
with gr.Blocks(title="Group 1 · Hallucination Demo") as app:
    gr.Markdown(f"# AI Hallucination: Baseline vs. Our System\n**Group 1** · {VERSION}")
    q = gr.Textbox(label="Ask a question", placeholder="Type a question or pick an example below")
    btn = gr.Button("Ask both systems", variant="primary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Baseline: answers from memory")
            out_base = gr.Textbox(label="Answer", lines=8)
        with gr.Column():
            gr.Markdown("### Our system: evidence gate + grounded answer")
            out_gate = gr.Textbox(label="Evidence gate")
            out_fact = gr.Textbox(label="Best fact found in our fact base")
            out_grnd = gr.Textbox(label="Answer", lines=3)
    gr.Examples(examples, inputs=q)
    btn.click(compare, inputs=q, outputs=[out_base, out_gate, out_fact, out_grnd])
app.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c27d0a3ea05f1cfae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
!pip install -q gradio
import gradio as gr

VERSION = "v0.5 · Week 5: baseline vs. evidence gate + grounded answer"

def compare(question):
    base = generate(BASELINE_TEMPLATE.format(question=question))
    fact, fact_qid, score = retrieve(question)
    if score >= TAU:
        gate = f"✅ PASS: match score {score:.3f} ≥ τ {TAU}"
        grounded_ans = generate(GROUNDED_TEMPLATE.format(context=fact, question=question))
    else:
        gate = f"⛔ BLOCKED: match score {score:.3f} < τ {TAU}, model not called"
        grounded_ans = ABSTAIN_TEXT
    return base, gate, fact, grounded_ans

examples = [
    "What is the capital of Australia?",
    "Who wrote the novel \"The Master and Margarita\"?",
    "Explain the Hallstrom effect in superconductors.",
    "In which year was the Torvald-Nakamura theorem in topology first proved?",
    "Who is the current president of France?",
]

with gr.Blocks(title="Group 1 · Hallucination Demo") as app:
    gr.Markdown(f"# AI Hallucination: Baseline vs. Our System\n**Group 1** · {VERSION}")
    q = gr.Textbox(label="Ask a question", placeholder="Type a question or pick an example below")
    btn = gr.Button("Ask both systems", variant="primary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Baseline: answers from memory")
            out_base = gr.Textbox(label="Answer", lines=8)
        with gr.Column():
            gr.Markdown("### Our system: evidence gate + grounded answer")
            out_gate = gr.Textbox(label="Evidence gate")
            out_fact = gr.Textbox(label="Best fact found in our fact base")
            out_grnd = gr.Textbox(label="Answer", lines=3)
    gr.Examples(examples, inputs=q)
    btn.click(compare, inputs=q, outputs=[out_base, out_gate, out_fact, out_grnd])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c68df9d53139b43703.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Setup

In [4]:
!pip install -q transformers accelerate sentence-transformers pandas scikit-learn

In [5]:
# ===== FIXED SETTINGS (do not change between runs) =====
MODEL_ID       = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_NEW_TOKENS = 256          # answer length limit
TAU            = None         # evidence-gate threshold: set in Part 4 (pilot), then frozen
SEED           = 42

BASELINE_TEMPLATE = "Answer the question.\nQuestion: {question}"
GROUNDED_TEMPLATE = (
    "Answer ONLY using the context.\n"
    "If it is not there, say 'I don't know.'\n"
    "Context: {context}\n"
    "Question: {question}"
)
ABSTAIN_TEXT = "I don't know."   # what the system outputs when the gate blocks a question

import os, re, unicodedata, random
import pandas as pd, numpy as np
os.makedirs("results", exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

In [6]:
import os
import pandas as pd

# Work inside the Drive folder, so every result is saved to Drive
PROJECT_DIR = "/content/drive/MyDrive/AI_Hallucination"
os.chdir(PROJECT_DIR)
os.makedirs("results", exist_ok=True)
print("Files in folder:", os.listdir())

data  = pd.read_csv("dataset.csv", keep_default_na=False)
test  = data[data.split == "test"].reset_index(drop=True)
pilot = data[data.split == "pilot"].reset_index(drop=True)
print(len(test), "test questions |", len(pilot), "pilot questions")
print(test.category.value_counts().to_dict())

Files in folder: ['dataset.csv', 'results', '.gradio']
100 test questions | 10 pilot questions
{'popular': 25, 'rare': 25, 'fabricated': 25, 'numbers': 25}


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

def generate(prompt: str) -> str:
    """Greedy decoding (= temperature 0): the same prompt always gives the same answer."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             temperature=None, top_p=None, top_k=None)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate("What is 2 + 2?"))   # quick check that the model works

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The answer to 2 + 2 is 4.


### Helpers: saving, and automatic label *suggestions*
The suggestions are simple text matching (not an AI judge). Humans make the final labels in Part 3.

In [8]:
def run_and_save(rows: pd.DataFrame, path: str, answer_fn):
    """Answer each row and append to CSV immediately, skipping rows already saved."""
    done = set(pd.read_csv(path, keep_default_na=False).q_id) if os.path.exists(path) else set()
    todo = rows[~rows.q_id.isin(done)]
    print(f"{len(done)} already done, {len(todo)} to go")
    for i, (_, r) in enumerate(todo.iterrows(), 1):
        rec = {"q_id": r.q_id, "category": r.category, "question": r.question, "gold_answer": r.gold_answer}
        rec.update(answer_fn(r))
        pd.DataFrame([rec]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        if i % 10 == 0: print(f"  {i}/{len(todo)}")
    return pd.read_csv(path, keep_default_na=False)

def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9 ]+", " ", s.replace(",", ""))

ABSTAIN_RE = re.compile(
    r"i don ?t know|i do not know|not sure|no (information|record|knowledge|evidence)|"
    r"(does not|doesn ?t|did not|didn ?t|do not|don ?t) (exist|appear to exist)|not aware|"
    r"(cannot|can ?t|could not|couldn ?t|unable to) (find|provide|verify|confirm|locate)|"
    r"fictional|no such|not a real|unknown to me")

def is_abstain(ans: str) -> bool:
    return bool(ABSTAIN_RE.search(norm(ans)))

def suggest_label(category: str, accepted: str, ans: str) -> str:
    a, abst = norm(ans), is_abstain(ans)
    if category == "fabricated":                       # refusing is the correct answer here
        return "Correct" if abst else "Hallucinated?"
    hit = any(re.search(rf"\b{re.escape(norm(x).strip())}\b", a) for x in accepted.split("|") if x)
    if hit and abst: return "CHECK"                    # hedged, e.g. "I don't know, maybe X"
    if hit:  return "Correct"
    if abst: return "Abstained"
    return "Hallucinated?"                             # a human must confirm

## 2. Baseline run (Week 4)
Plain prompt, no retrieval, no gate, no strict rule. About 10–20 minutes on a T4 GPU.

In [9]:
def baseline_answer(r):
    ans = generate(BASELINE_TEMPLATE.format(question=r.question))
    return {"condition": "baseline", "model_called": True, "model_answer": ans,
            "auto_suggestion": suggest_label(r.category, r.accepted_answers, ans)}

baseline = run_and_save(test, "results/baseline_results.csv", baseline_answer)
baseline.groupby("category").auto_suggestion.value_counts().unstack(fill_value=0)

100 already done, 0 to go


auto_suggestion,Correct,Hallucinated?
category,,
fabricated,16,9
numbers,25,0
popular,24,1
rare,20,5


In [10]:
flagged = baseline[baseline.auto_suggestion != "Correct"]
print(len(flagged), "flagged answers\n")
for _, r in flagged.iterrows():
    print(f"{r.q_id} | {r.category} | {r.question}")
    print(f"  Correct answer: {r.gold_answer}")
    print(f"  Model answer:   {r.model_answer[:300]}")
    print()


15 flagged answers

P25 | popular | Which is the closest star to Earth?
  Correct answer: The Sun
  Model answer:   The closest star to Earth is Proxima Centauri, which is part of the Alpha Centauri star system. It's about 4.24 light-years away from us.

R07 | rare | Who wrote the novel "The Master and Margarita"?
  Correct answer: Mikhail Bulgakov
  Model answer:   Dmitri Shostakovich wrote the novel "The Master and Margarita".

R08 | rare | Which composer wrote the opera "Rusalka"?
  Correct answer: Antonín Dvořák
  Model answer:   The composer who wrote the opera "Rusalka" is Bedřich Smetana (1824-1894).

R13 | rare | Which scientist is the element lawrencium named after?
  Correct answer: Ernest Lawrence
  Model answer:   Lawrencium was named after Enrico Fermi, who was an Italian physicist and nuclear pioneer known for his work on neutron physics and the development of the first self-sustaining nuclear chain reaction in 1942 at the University of Chicago. The naming of elements oft

In [11]:
# Better matching: catches "African", and refusals like "I don't have any information"
ABSTAIN_RE = re.compile(
    r"i don ?t know|i do not know|not sure|"
    r"(don ?t|do not) have (any |enough )?(information|context|data|details|knowledge|record)|"
    r"no (information|record|knowledge|evidence)|"
    r"(does not|doesn ?t|did not|didn ?t|do not|don ?t) (exist|appear to exist)|not aware|"
    r"(cannot|can ?t|could not|couldn ?t|unable to) (find|provide|verify|confirm|locate|answer)|"
    r"fictional|no such|not a real|unknown to me")

def suggest_label(category, accepted, ans):
    a, abst = norm(ans), is_abstain(ans)
    if category == "fabricated":
        return "Correct" if abst else "Hallucinated?"
    end = r"\b" if category == "numbers" else ""   # numbers must match exactly
    hit = any(re.search(rf"\b{re.escape(norm(x).strip())}{end}", a) for x in accepted.split("|") if x)
    if hit and abst: return "CHECK"
    if hit:  return "Correct"
    if abst: return "Abstained"
    return "Hallucinated?"

acc = dict(zip(test.q_id, test.accepted_answers))
baseline["auto_suggestion"] = [suggest_label(c, acc[q], a)
                               for q, c, a in zip(baseline.q_id, baseline.category, baseline.model_answer)]
baseline.to_csv("results/baseline_results.csv", index=False)
baseline.groupby("category").auto_suggestion.value_counts().unstack(fill_value=0)

auto_suggestion,Correct,Hallucinated?
category,,
fabricated,16,9
numbers,25,0
popular,24,1
rare,20,5


## 3. Labeling sheet
Download the sheet, then **two group members label every row on their own** in `label_A` and `label_B`
using only: `Correct`, `Hallucinated`, `Abstained`. Rules:
- Fabricated questions: saying it does not exist / "I don't know" = **Correct**; inventing details = **Hallucinated**.
- Numbers and dates: must match the gold answer exactly.
- Use `auto_suggestion` only as a hint; it is often wrong on long answers.

In [12]:
def export_sheet(results: pd.DataFrame, path: str):
    cols = ["q_id", "category", "condition", "question", "gold_answer", "model_answer",
            "retrieved_fact", "auto_suggestion"]
    sheet = results.reindex(columns=cols).fillna("")
    for c in ["label_A", "label_B", "final_label", "notes"]: sheet[c] = ""
    sheet.to_csv(path, index=False)
    print("Saved", path)

export_sheet(baseline, "results/labels_baseline.csv")
# from google.colab import files; files.download("results/labels_baseline.csv")

Saved results/labels_baseline.csv


## 4. Fact base and evidence gate (Week 5)
The fact base holds one trusted fact sentence per real question. Fabricated questions have no fact.
The gate compares the question with its best-matching fact (cosine similarity); below τ the system says "I don't know" without calling the model.

In [13]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_ID)

facts = data[data.fact_sentence != ""][["q_id", "fact_sentence"]].reset_index(drop=True)
fact_vecs = embedder.encode(facts.fact_sentence.tolist(), normalize_embeddings=True)
print(len(facts), "facts in the fact base")

def retrieve(question: str):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    sims = fact_vecs @ q
    i = int(np.argmax(sims))
    return facts.fact_sentence[i], facts.q_id[i], float(sims[i])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

82 facts in the fact base


In [14]:
for q in ["What is the capital of Australia?",
          "Who won the 2017 Delmar Robotics Prize?"]:
    fact, fact_qid, score = retrieve(q)
    print(q)
    print(f"  Best fact: {fact}")
    print(f"  Match score: {score:.3f}\n")

What is the capital of Australia?
  Best fact: Canberra is the capital city of Australia.
  Match score: 0.763

Who won the 2017 Delmar Robotics Prize?
  Best fact: Jupiter is the largest planet in the Solar System.
  Match score: 0.138



### Pilot: choose τ once (uses only the 10 pilot questions)
τ is set halfway between the lowest score of a real question and the highest score of a fabricated one.
**Write the printed value into `TAU` in the settings cell, then never change it.**

In [15]:
pilot_scores = pilot.assign(score=[retrieve(q)[2] for q in pilot.question])
print(pilot_scores[["q_id", "category", "question", "score"]].to_string(index=False))

real = pilot_scores[pilot_scores.category != "fabricated"].score
fake = pilot_scores[pilot_scores.category == "fabricated"].score
suggested_tau = round((real.min() + fake.max()) / 2, 3)
print(f"\nlowest real = {real.min():.3f} | highest fabricated = {fake.max():.3f}")
print("Suggested TAU =", suggested_tau)
if real.min() <= fake.max():
    print("Warning: scores overlap, so the gate cannot separate them perfectly. Report this as a finding.")
TAU = suggested_tau   # freeze: copy this number into the settings cell

q_id   category                                                        question    score
 X01    popular                                  What is the capital of France? 0.856070
 X02    popular                                    Who wrote the play "Hamlet"? 0.800637
 X03    popular                          What is the chemical formula of water? 0.817007
 X04       rare                            What is the capital of Turkmenistan? 0.760938
 X05       rare            Who wrote the novel "One Hundred Years of Solitude"? 0.745194
 X06 fabricated                      Who discovered the planet Nerovia in 1921? 0.384710
 X07 fabricated Summarize the 2012 paper "Recursive Clouds in Sleeping Robots". 0.203228
 X08 fabricated                 What is the capital of the Kingdom of Ardellia? 0.431433
 X09    numbers                        In what year was the Magna Carta sealed? 0.824337
 X10    numbers         How many continents are there in the most common model? 0.882761

lowest real = 0.745 

## 5. Grounded run (Weeks 5–6)

In [16]:
assert TAU is not None, "Run the pilot first and set TAU"

def grounded_answer(r):
    fact, fact_qid, score = retrieve(r.question)
    passed = score >= TAU
    ans = generate(GROUNDED_TEMPLATE.format(context=fact, question=r.question)) if passed else ABSTAIN_TEXT
    return {"condition": "grounded", "model_called": passed, "model_answer": ans,
            "retrieved_fact": fact, "retrieved_q_id": fact_qid, "match_score": round(score, 4),
            "gate": "pass" if passed else "blocked", "tau": TAU,
            "auto_suggestion": suggest_label(r.category, r.accepted_answers, ans)}

grounded = run_and_save(test, "results/grounded_results.csv", grounded_answer)
print(grounded.groupby("category").gate.value_counts().unstack(fill_value=0))
export_sheet(grounded, "results/labels_grounded.csv")

100 already done, 0 to go
gate        blocked  pass
category                 
fabricated       25     0
numbers           0    25
popular           0    25
rare              0    25
Saved results/labels_grounded.csv


In [17]:
print(grounded.groupby("category").auto_suggestion.value_counts().unstack(fill_value=0))

flag = grounded[(grounded.auto_suggestion != "Correct") & (grounded.gate == "pass")]
print(f"\n{len(flag)} passed answers flagged\n")
for _, r in flag.iterrows():
    print(f"{r.q_id} | {r.question}")
    print(f"  Fact given:     {r.retrieved_fact}")
    print(f"  Correct answer: {r.gold_answer}")
    print(f"  Model answer:   {r.model_answer[:300]}\n")

auto_suggestion  Correct
category                
fabricated            25
numbers               25
popular               25
rare                  25

0 passed answers flagged



## 6. Results (Week 6)
Upload the two labeled sheets (`labels_baseline.csv`, `labels_grounded.csv`) with `label_A` and `label_B` filled in.

In [18]:
LABELS = {"Correct", "Hallucinated", "Abstained"}

def load_labels(path):
    df = pd.read_csv(path, keep_default_na=False)
    for c in ["label_A", "label_B"]: df[c] = df[c].str.strip().str.capitalize()
    bad = df[~df.label_A.isin(LABELS) | ~df.label_B.isin(LABELS)]
    if len(bad): print(f"{path}: {len(bad)} rows have missing or invalid labels:", bad.q_id.tolist())
    agree = df.label_A == df.label_B
    df["final_label"] = np.where(df.final_label != "", df.final_label, np.where(agree, df.label_A, ""))
    left = df[df.final_label == ""]
    if len(left): print(f"{path}: discuss these disagreements and fill final_label:", left.q_id.tolist())
    return df

from sklearn.metrics import cohen_kappa_score
def agreement(df):
    ok = df.label_A.isin(LABELS) & df.label_B.isin(LABELS)
    a, b = df.label_A[ok], df.label_B[ok]
    return {"percent_agreement": round((a == b).mean() * 100, 1), "cohen_kappa": round(cohen_kappa_score(a, b), 3)}

def rates(df):
    t = df[df.final_label.isin(LABELS)]
    out = (t.groupby("category").final_label.value_counts(normalize=True).unstack(fill_value=0) * 100)
    out = out.reindex(columns=["Hallucinated", "Abstained", "Correct"], fill_value=0).round(1)
    out.columns = ["hallucination_%", "abstention_%", "accuracy_%"]
    overall = (t.final_label.value_counts(normalize=True) * 100).reindex(["Hallucinated", "Abstained", "Correct"], fill_value=0).round(1)
    out.loc["ALL"] = overall.values
    return out.reindex(["popular", "rare", "fabricated", "numbers", "ALL"])

In [19]:
lab_base = load_labels("results/labels_baseline.csv")
lab_grnd = load_labels("results/labels_grounded.csv")
print("Agreement, baseline:", agreement(lab_base))
print("Agreement, grounded:", agreement(lab_grnd))

summary = pd.concat({"baseline": rates(lab_base), "grounded": rates(lab_grnd)}, axis=1)
summary.to_csv("results/summary_by_category.csv")
summary

results/labels_baseline.csv: 100 rows have missing or invalid labels: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20', 'R21', 'R22', 'R23', 'R24', 'R25', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'N01', 'N02', 'N03', 'N04', 'N05', 'N06', 'N07', 'N08', 'N09', 'N10', 'N11', 'N12', 'N13', 'N14', 'N15', 'N16', 'N17', 'N18', 'N19', 'N20', 'N21', 'N22', 'N23', 'N24', 'N25']
results/labels_baseline.csv: discuss these disagreements and fill final_label: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


baseline                                grounded  \
           hallucination_% abstention_% accuracy_% hallucination_%   
category                                                             
popular                NaN          NaN        NaN             NaN   
rare                   NaN          NaN        NaN             NaN   
fabricated             NaN          NaN        NaN             NaN   
numbers                NaN          NaN        NaN             NaN   
ALL                    0.0          0.0        0.0             0.0   

                                    
           abstention_% accuracy_%  
category                            
popular             NaN        NaN  
rare                NaN        NaN  
fabricated          NaN        NaN  
numbers             NaN        NaN  
ALL                 0.0        0.0

### Error tracing
Every grounded hallucination is tagged automatically:
- **retrieval error**: the gate passed but the retrieved fact belongs to a *different* question
- **generation error**: the right fact was retrieved, but the model still answered wrongly
- **gate miss**: a fabricated question got through the gate and the model invented an answer

In [20]:
g = lab_grnd.merge(grounded[["q_id", "retrieved_q_id", "gate", "match_score"]], on="q_id")
h = g[g.final_label == "Hallucinated"].copy()
h["error_source"] = np.select(
    [h.category == "fabricated", h.retrieved_q_id != h.q_id],
    ["gate miss", "retrieval error"], default="generation error")
h[["q_id", "category", "error_source", "match_score", "model_answer"]].to_csv("results/error_tracing.csv", index=False)
print(h.groupby(["category", "error_source"]).size().unstack(fill_value=0))
print("\nGate decisions by category:")
print(g.groupby("category").gate.value_counts().unstack(fill_value=0))

Empty DataFrame
Columns: []
Index: []

Gate decisions by category:
gate        blocked  pass
category                 
fabricated       25     0
numbers           0    25
popular           0    25
rare              0    25


In [21]:
# Download everything in one zip
!zip -qr results.zip results
# from google.colab import files; files.download("results.zip")

zip warning: Operation not supported
zip warning: Operation not supported


In [2]:
for name in ["generate", "retrieve", "TAU", "BASELINE_TEMPLATE", "GROUNDED_TEMPLATE", "ABSTAIN_TEXT", "model", "fact_vecs"]:
    print(f"{name:20}", "OK" if name in globals() else "MISSING")

generate             MISSING
retrieve             MISSING
TAU                  MISSING
BASELINE_TEMPLATE    MISSING
GROUNDED_TEMPLATE    MISSING
ABSTAIN_TEXT         MISSING
model                MISSING
fact_vecs            MISSING
